In [2]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"

In [3]:
Artist_name ="Pierre-Auguste Renoir"

In [4]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [5]:
gemini_label.shape

(1000, 10)

In [6]:
full_df = claude_label.merge(gemini_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_gemini"))

In [7]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_gemini,artistic_value_comment_gemini,creativity_answer_gemini,creativity_comment_gemini
0,41,Baigneuse,Pierre-Auguste,Renoir,1888,French,High,Impressionist artists Berthe Morisot and Claud...,Yes,"At the beginning of the 1880s, Renoir felt he ...",High,"Renoir's ""Baigneuse"" (1888) is widely regarded...",Yes,"Renoir's ""Baigneuse"" (1888) demonstrates creat..."
1,62,"Femme au peplum rouge, tête, bras",Pierre-Auguste,Renoir,1895,French,scholarly commentary on this Renoir painting f...,"To properly answer your question, I would need...",for scholarly commentary on this Renoir painti...,"To properly answer your question, I would need...",High,"Pierre-Auguste Renoir's ""Femme au peplum rouge...",Yes,"""Femme au peplum rouge, tête, bras"" demonstrat..."
2,149,Saule au bord d'une mare,Pierre-Auguste,Renoir,1874,French,Unable to Determine,Authoritative scholarly commentary on this spe...,Unable to Determine,Without authoritative critical sources specifi...,High,"Pierre-Auguste Renoir's ""Saule au bord d'une m...",Yes,"""Saule au bord d'une mare"" is a creative artwo..."
3,628,Compotier de fruits,Pierre-Auguste,Renoir,1890,French,"authoritative commentary on Renoir's ""Compotie...","To obtain this analysis, I recommend:**\n- Con...","for authoritative commentary on Renoir's ""Comp...","To obtain this analysis, I recommend:**\n- Con...",High,Pierre-Auguste Renoir's 'Compotier de fruits' ...,Yes,'Compotier de fruits' (1890) demonstrates crea...
4,1184,Paysage,Pierre-Auguste,Renoir,1917,French,High,"The 1917 ""Paysage"" is part of the Barnes Found...",No,While Renoir's 1917 works represent a stylisti...,High,"Pierre-Auguste Renoir's 1917 ""Paysage"" (Landsc...",Yes,"Renoir's ""Paysage"" from 1917 demonstrates crea..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,426721547,Le chapeau épinglé,Pierre-Auguste,Renoir,1894,French,High,The work depicts a serene view of two young wo...,Yes,Renoir treated this scene of two young girls s...,High,"""Le Chapeau Épinglé"" by Pierre-Auguste Renoir ...",Yes,"""Le Chapeau Épinglé"" demonstrates considerable..."
996,426721548,Baigneuse assise,Pierre-Auguste,Renoir,1897,French,High,The work represents a culmination of Renoir's ...,Yes,The painting is composed using contradictory a...,High,"Pierre-Auguste Renoir's ""Baigneuse assise"" (18...",Yes,"""Baigneuse assise"" demonstrates creativity thr..."
997,426721549,Le chapeau épinglé,Pierre-Auguste,Renoir,1898,French,High,This extremely delicate work expresses the bea...,Yes,By the 1890s lithography was becoming reaccept...,High,"""Le Chapeau épinglé"" by Pierre-Auguste Renoir ...",Yes,"""Le Chapeau épinglé"" demonstrates significant ..."
998,426721550,"Étude de femme nue, assise, variante (from L'a...",Pierre-Auguste,Renoir,1904,French,scholarly commentary on this specific Renoir w...,Recommendation:** I suggest consulting:\n- The...,for scholarly commentary on this specific Reno...,Recommendation:** I suggest consulting:\n- The...,High,"The artwork ""Étude de femme nue, assise, varia...",Yes,"""Étude de femme nue, assise, variante"" demonst..."


In [8]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [10]:
claude_embed = np.load(f"clip_embeddings_claude_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
gemini_embed = np.load(f"clip_embeddings_gemini_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
openai_embed = np.load(f"clip_embeddings_openai_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)

# Golden Set

Golden set is a set of samples that have the same answers for "type" and "creative". To add to confidence, only samples with cosine similarity above a given threshold are kept.

In [71]:
threshold = 0.70

In [72]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [73]:
np.sum(consistent_artist)

np.int64(710)

In [74]:
np.sum(consistent_creative)

np.int64(305)

In [75]:
np.sum(consistent_overall)

np.int64(305)

In [76]:
checking=full_df.copy()
checking["creative_consist"]=consistent_creative
checking["artistic_consist"]=consistent_artist
checking["overall_consist"]=consistent_overall

In [77]:
print(f"""
For type, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
""")

print(f"""
For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
""")


For type, there are 710 samples that are consistent.


For creative, there are 305 samples that are consistent.



In [78]:
print(f"""
When looking at only type and creative, there are {np.sum(checking["overall_consist"])} samples that are consistent.
""")


When looking at only type and creative, there are 305 samples that are consistent.



In [79]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > threshold).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > threshold).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: End")

2025-12-16 04:56:02: Start
2025-12-16 04:56:02: Currently at 0
2025-12-16 04:56:02: End


In [80]:
checking["embed_artistic_consist"]=embed_consistent_artistic
checking["embed_creative_consist"]=embed_consistent_creative
checking["embed_overall_consist"]=embed_consistent_overall

In [81]:
print(f"""
[Easy] For artistic, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
[Hard] For artistic, there are {checking[(checking["artistic_consist"]==1) & (checking["embed_artistic_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy] For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
[Hard] For creative, there are {checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy]For overall, there are {np.sum(checking["overall_consist"])} samples that are consistent.
[Hard] For overall, there are {checking[(checking["overall_consist"]==1) & (checking["embed_overall_consist"]==1)].shape[0]} samples that are consistent.
""")


[Easy] For artistic, there are 710 samples that are consistent.
[Hard] For artistic, there are 452 samples that are consistent.


[Easy] For creative, there are 305 samples that are consistent.
[Hard] For creative, there are 189 samples that are consistent.


[Easy]For overall, there are 305 samples that are consistent.
[Hard] For overall, there are 130 samples that are consistent.



In [82]:
golden_set_move_creative = checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)]

In [83]:
golden_set_move_creative.shape

(189, 24)

In [84]:
golden_set_move_creative.to_excel(f"golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx",index=False)

# Double Check

In [22]:
golden_set_move_creative = pd.read_excel("golden_set_move_creative_65.xlsx")

In [23]:
golden_set_move_creative

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,...,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment,creative_consist,artistic_consist,overall_consist,embed_artistic_consist,embed_creative_consist,embed_overall_consist
0,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,...,High,"""Le Peintre et Son Modèle"" is a significant wo...",Yes,"In ""Le Peintre et Son Modèle,"" Picasso innovat...",1,1,1,1,1,1
1,1781,Verre et citron,Pablo,Picasso,1944,Spanish,High,"According to art historian John Richardson, ""t...",Yes,The work belongs to a series of small-scale wo...,...,High,"""Verre et citron"" is a notable example of Pica...",Yes,"""Verre et citron"" exemplifies Picasso's innova...",1,1,1,1,1,1
2,2849,HOMME ASSIS,Pablo,Picasso,1969,Spanish,High,"""Homme Assis"" was painted during Picasso's mos...",Yes,Picasso's objective to paint 'nature' contrast...,...,High,"Pablo Picasso's ""Homme Assis"" (1969) is a sign...",Yes,"""Homme Assis"" exemplifies Picasso's innovative...",1,1,1,1,1,1
3,3100,"Femme assise dans un fauteuil tressé, en gris ...",Pablo,Picasso,1953,Spanish,High,This portrait of Françoise Gilot was painted i...,Yes,The 1953 portrait innovates beyond Picasso's e...,...,High,"""Femme assise dans un fauteuil tressé, en gris...",Yes,"Picasso's ""Femme assise dans un fauteuil tress...",1,1,1,0,1,0
4,3483,DEUX HIRONDELLES,Pablo,Picasso,1932,Spanish,High,"Painted on May 14, 1932 at the height of his c...",Yes,The work demonstrates creativity through surpr...,...,High,"Pablo Picasso's 1932 painting ""Deux Hirondelle...",Yes,"""Deux Hirondelles"" exemplifies Picasso's creat...",1,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1741,424922090,Visage de femme,Pablo,Picasso,1953,Spanish,High,"This ceramic work, likely depicting Jacqueline...",Yes,"Picasso was inspired by those around him, with...",...,High,"""Visage de femme"" (1953) is a glazed ceramic p...",Yes,"""Visage de femme"" demonstrates Picasso's creat...",1,1,1,1,1,1
1742,424922091,Visage d'homme,Pablo,Picasso,1953,Spanish,High,While specific scholarly critique of this 1953...,Yes,Picasso's use of the cast shadow as a pictoria...,...,High,"""Visage d'homme"" (1953) exemplifies Picasso's ...",Yes,"""Visage d'homme"" showcases Picasso's continuou...",1,1,1,1,1,1
1743,424924633,Vase aztèque aux quatre visages,Pablo,Picasso,1957,Spanish,High,Vase Aztèque aux quatre visages captures the s...,Yes,This work captures Picasso's restless need to ...,...,High,"Pablo Picasso's ""Vase Aztèque aux Quatre Visag...",Yes,"Picasso's ""Vase Aztèque aux Quatre Visages"" de...",1,1,1,1,1,1
1744,424927848,Mousquetaire,Pablo,Picasso,1969,Spanish,High,"Between 1966 and 1972, Picasso displayed porte...",Yes,"For Picasso, the musketeer signified the golde...",...,High,"Pablo Picasso's 1969 painting ""Mousquetaire"" e...",Yes,"In ""Mousquetaire,"" Picasso showcases significa...",1,1,1,1,1,1
